In [107]:
import pandas as pd

In [108]:
locations_df = pd.read_csv("../data/locations.csv")
providers_df = pd.read_csv("../data/providers.csv")
payers_df = pd.read_csv("../data/payers.csv")
patients_df = pd.read_csv("../data/patients.csv")
calls_df = pd.read_csv("../data/calls.csv")
appointments_df = pd.read_csv("../data/appointments.csv")

for name, df in [("locations", locations_df), ("providers", providers_df),
                  ("payers", payers_df), ("patients", patients_df),
                  ("calls", calls_df), ("appointments", appointments_df)]:
    print(name, df.shape)

locations (4, 4)
providers (8, 5)
payers (6, 3)
patients (9439, 4)
calls (28493, 6)
appointments (64534, 12)


In [109]:
appointments_df.head()

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
0,A1,2023-08-01,2023-07-23,P1,L1,PT1,PY1,Follow-up,False,Completed,92.03,0.93
1,A2,2023-08-01,2023-07-27,P1,L1,PT1,PY1,Physical Therapy,False,Cancelled,0.00,0.00
2,A3,2023-08-01,2023-08-01,P1,L1,PT1,PY1,Post-Op Check,False,Completed,75.98,0.57
3,A4,2023-08-01,2023-07-26,P1,L1,PT1,PY1,Follow-up,False,Completed,130.28,0.68
4,A5,2023-08-01,2023-07-30,P1,L1,PT1,PY1,Follow-up,False,Completed,137.98,1.03


In [110]:
appointments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64534 entries, 0 to 64533
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   appointment_id    64534 non-null  object 
 1   date              64534 non-null  object 
 2   booked_date       64534 non-null  object 
 3   provider_id       64534 non-null  object 
 4   location_id       64534 non-null  object 
 5   patient_id        64534 non-null  object 
 6   payer_id          64534 non-null  object 
 7   appointment_type  64534 non-null  object 
 8   is_new_patient    64534 non-null  bool   
 9   status            64534 non-null  object 
 10  revenue           64012 non-null  float64
 11  rvu               64534 non-null  float64
dtypes: bool(1), float64(2), object(9)
memory usage: 5.5+ MB


In [111]:
# Convert dates to datetime format
appointments_df['date'] = pd.to_datetime(appointments_df['date'], errors='coerce', format='%Y-%m-%d')
appointments_df['booked_date'] = pd.to_datetime(appointments_df['booked_date'], errors='coerce', format='%Y-%m-%d')

In [112]:
# Clean column names by stripping whitespace and converting to lowercase
appointments_df.columns = appointments_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
appointments_df.columns.tolist()

['appointment_id',
 'date',
 'booked_date',
 'provider_id',
 'location_id',
 'patient_id',
 'payer_id',
 'appointment_type',
 'is_new_patient',
 'status',
 'revenue',
 'rvu']

In [113]:
appointments_df.isna().sum()

appointment_id        0
date                  0
booked_date           0
provider_id           0
location_id           0
patient_id            0
payer_id              0
appointment_type      0
is_new_patient        0
status                0
revenue             522
rvu                   0
dtype: int64

In [120]:
blank_revenue_df = appointments_df[(appointments_df['revenue'].isna()) & (appointments_df['status'] == 'Completed')]
blank_revenue_df.shape

(518, 12)

In [115]:
appointments_df[(appointments_df['revenue'] == 0) & (appointments_df['status'] == 'Completed')]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [116]:
bad_dates = appointments_df['booked_date'] > appointments_df['date']
appointments_df.loc[bad_dates, 'booked_date'] = pd.NaT

appointments_df['booked_date'].value_counts()

booked_date
2026-07-09    111
2026-06-23    107
2025-10-31    105
2026-07-02    105
2026-02-20    104
             ... 
2023-07-15      2
2023-07-12      2
2023-07-10      1
2023-07-11      1
2023-07-06      1
Name: count, Length: 1119, dtype: int64

In [117]:
# Drop duplicate rows based on the 'appointment_id' column
appointments_df.drop_duplicates(subset=['appointment_id'], keep='first', inplace=True)
appointments_df.shape # 64534 -> 64213

(64213, 12)

In [ ]:
set(providers_df['provider_id'])
provider_error_appointments_df = appointments_df[~appointments_df['provider_id'].isin(set(providers_df['provider_id']))]
provider_error_appointments_df

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
689,A690,2023-08-16,2023-08-10,P99,L2,PT67,PY3,Post-Op Check,False,Completed,87.12,0.76
1056,A1057,2023-08-24,2023-07-31,P99,L3,PT156,PY3,New Patient Consult,True,No-Show,0.00,0.00
2262,A2263,2023-09-21,2023-09-18,P99,L1,PT227,PY4,Physical Therapy,False,Completed,123.63,1.32
2883,A2884,2023-10-05,2023-10-05,P99,L1,PT33,PY5,Physical Therapy,False,Completed,124.15,1.40
3138,A3139,2023-10-10,2023-10-02,P99,L3,PT250,PY2,Follow-up,False,Completed,89.65,0.83
...,...,...,...,...,...,...,...,...,...,...,...,...
61217,A61218,2026-06-24,2026-06-12,P99,L1,PT4486,PY1,Follow-up,False,Completed,125.67,0.83
61387,A61388,2026-06-25,2026-06-24,P99,L4,PT3169,PY3,Follow-up,False,Completed,79.03,1.01
62024,A62025,2026-07-03,2026-06-18,P99,L1,PT5714,PY6,Follow-up,False,Completed,119.12,1.08
63150,A63151,2026-07-17,2026-07-06,P99,L2,PT4338,PY1,Follow-up,False,Completed,89.56,0.86


In [93]:
# P99 doesn't exist in providers_df, but it does exist in appointments_df. This is likely a data quality issue that needs to be addressed.
set(appointments_df['provider_id'])

{'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P99'}

In [122]:
invalid_provider_mask = ~appointments_df['provider_id'].isin(set(providers_df['provider_id']))
appointments_df.loc[invalid_provider_mask, 'provider_id'] = 'UNK'

appointments_df[appointments_df['provider_id'] == 'UNK']

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
689,A690,2023-08-16,2023-08-10,UNK,L2,PT67,PY3,Post-Op Check,False,Completed,87.12,0.76
1056,A1057,2023-08-24,2023-07-31,UNK,L3,PT156,PY3,New Patient Consult,True,No-Show,0.00,0.00
2262,A2263,2023-09-21,2023-09-18,UNK,L1,PT227,PY4,Physical Therapy,False,Completed,123.63,1.32
2883,A2884,2023-10-05,2023-10-05,UNK,L1,PT33,PY5,Physical Therapy,False,Completed,124.15,1.40
3138,A3139,2023-10-10,2023-10-02,UNK,L3,PT250,PY2,Follow-up,False,Completed,89.65,0.83
...,...,...,...,...,...,...,...,...,...,...,...,...
61217,A61218,2026-06-24,2026-06-12,UNK,L1,PT4486,PY1,Follow-up,False,Completed,125.67,0.83
61387,A61388,2026-06-25,2026-06-24,UNK,L4,PT3169,PY3,Follow-up,False,Completed,79.03,1.01
62024,A62025,2026-07-03,2026-06-18,UNK,L1,PT5714,PY6,Follow-up,False,Completed,119.12,1.08
63150,A63151,2026-07-17,2026-07-06,UNK,L2,PT4338,PY1,Follow-up,False,Completed,89.56,0.86


In [94]:
appointments_df[~appointments_df['location_id'].isin(set(locations_df['location_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [95]:
appointments_df[~appointments_df['patient_id'].isin(set(patients_df['patient_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [96]:
appointments_df[~appointments_df['payer_id'].isin(set(payers_df['payer_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [101]:
appointments_df.shape

(64020, 12)